In [25]:
import json

file_path = "/Users/ben/Documents/bruegel/data_new/WORKING/INDUSTRY/tuone_v6/src/cached_article_database.jsonl"

# Read and store JSONL data
articles = []
with open(file_path, "r", encoding="utf-8") as file:
    for i, line in enumerate(file):
        data = json.loads(line)
        articles.append(data)

        # Preview only the first 5 lines
        if i < 4:
            print(f"Line {i+1}: {data}")

print(f"\n✅ Loaded {len(articles)} articles in total.")

Line 1: {'title': 'Mitsubishi Chemical expands production capacities', 'paragraphs': {'p1': 'Tokyo-based Mitsubishi Chemical wants to boost up its battery electrolyte business. It plans several strategic steps due to growing demand on the electric vehicle market.', 'p2': 'The company plans to restarting production in a factory based in Britain and doubling its output in the States. Part of the reconstruction plan is also to close a factory in China. The company is said to be even willing to temporarily limit its mobile device battery production.', 'p3': 'Initially, the UK plant has built batteries since 2012. However, the anticipated demand did not arise, so the production lines were stopped in March 2016.asia.nikkei.com', 'p4': 'Your email address will not be published.Required fields are marked*', 'p5': 'Name *', 'p6': 'E-Mail *', 'p7': 'I agree with thePrivacy policy', 'p8': "We have been covering the development of electric mobility with journalistic passion and expertise since 201

In [26]:
import pandas as pd
df = pd.DataFrame(articles)

In [27]:
processed_articles = []
for raw_data in articles:
    # extracting the paragraphs
    paragraphs = [raw_data[key] for key in sorted(raw_data.keys()) if key.startswith("p")]

    #extract the meta-data object
    meta_data = raw_data.get("meta", {})

    # Only process articles where ID starts with "27"
    if meta_data.get("ID", "").startswith("27"):
        article = {
            "_id": raw_data["_id"],
            "title": raw_data.get("title", ""),
            "paragraphs": paragraphs,  # Store paragraphs as a list
            "meta": {
                "ID": meta_data.get("ID", ""),
                "date": meta_data.get("date", ""),
                "url": meta_data.get("url", "")
            }
        }

        processed_articles.append(article)

# Display sample processed articles
display(pd.DataFrame(processed_articles).head())

,title,paragraphs,meta
0,"United Kingdom, Honda + GM, Delaware, Milton K...",[{'p1': '75m for electric transport: The Briti...,"{'ID': '2700008', 'date': '18-01-2016', 'url':..."
1,"BAIC, China, Tesla, Apple, Spyker, Volvo.",[{'p1': 'BAIC wants to increase EV production:...,"{'ID': '2700013', 'date': '26-01-2016', 'url':..."
2,"BMW, Wanxiang, Faraday Future, Honda, Mercedes...",[{'p1': 'BMW enters power storage market: The ...,"{'ID': '2700025', 'date': '25-01-2016', 'url':..."
3,"Morgan, Toyota, EV uptake, BMW, PowerCell Swed...",[{'p1': 'More Morgan models: British Morgan Mo...,"{'ID': '2700027', 'date': '20-01-2016', 'url':..."
4,"Volkswagen, Icona, Fiat Chrysler, Toyota, Kand...",[{'p1': 'VW to electrify China: Volkswagen has...,"{'ID': '2700032', 'date': '29-01-2016', 'url':..."


In [28]:
if processed_articles:
    print(json.dumps(processed_articles[0], indent=4))


{
    "title": "United Kingdom, Honda + GM, Delaware, Milton Keynes.",
    "paragraphs": [
        {
            "p1": "75m for electric transport: The British government and industry awarded almost 75m pounds to five projects. The biggest bulk (46.5m GBP) goes to the London Taxi Corporation to develop electric cabs. Morgan Motors receives funding to develop hybrid- and electric drives for sports cars, while AGM Batteries is working on battery packs. Parker Hannifin looks at electric forklifts and last, Jaguar Land Rover receives 13.1m GBP for turbocharger supply in theUK.",
            "p2": "gov.uk",
            "p3": "Joint fuel cell production: Honda and GM will set up a joint fuel cell production facility. The firms have been working together on the technology since 2013 and aim for a mass market eco car. The move is expected to decrease costs. Serial production of fuel cells will start by 2025 at the latest.",
            "p4": "asahi.com",
            "p5": "Fuel cell price brea

In [29]:
print(f"Total articles to process: {len(articles)}")
print(f"Total processed articles: {len(processed_articles)}")


Total articles to process: 38310
Total processed articles: 2816


In [30]:
#"meta.ID": {"$regex": "^27"}

In [8]:
import os
import certifi
from pymongo import MongoClient
from pymongo.server_api import ServerApi
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get MongoDB credentials from environment variables
MONGO_URI = os.getenv("MONGO_URI")
DB_NAME = os.getenv("MONGO_DB_NAME")
COLLECTION_NAME = os.getenv("ARTICLES_COLLECTION_NAME")

# Ensure all necessary environment variables are set
if not all([MONGO_URI, DB_NAME, COLLECTION_NAME]):
    print("❌ Missing required environment variables. Check your .env file.")
    exit()

# Create MongoDB client with TLS verification
try:
    client = MongoClient(MONGO_URI, server_api=ServerApi('1'), tlsCAFile=certifi.where())
    db = client[DB_NAME]
    collection = db[COLLECTION_NAME]

    # Verify connection
    client.admin.command('ping')
    print("✅ Connected to MongoDB Atlas!")

except Exception as e:
    print(f"❌ MongoDB Connection Error: {e}") 
    exit()

Python-dotenv could not parse statement starting at line 1
Python-dotenv could not parse statement starting at line 2
Python-dotenv could not parse statement starting at line 3
Python-dotenv could not parse statement starting at line 4
Python-dotenv could not parse statement starting at line 5
Python-dotenv could not parse statement starting at line 6
Python-dotenv could not parse statement starting at line 7


✅ Connected to MongoDB Atlas!


In [32]:
import time

total_records = len(processed_articles)
success_count = 0
failures = []

print("\n🚀 Starting data insertion into MongoDB...\n")

for i, article in enumerate(processed_articles, start=1):
    try:
        collection.insert_one(article)
        success_count += 1
        print(f"✅ Processing {i} of {total_records}... Inserted successfully.")
    except Exception as e:
        print(f"❌ Error inserting record {i}: {e}")
        failures.append({"index": i, "error": str(e)})
    time.sleep(0.1)  # Small delay to avoid overwhelming MongoDB (optional)

# Final summary
print(f"\n✅ Successfully inserted {success_count} out of {total_records} records into MongoDB Atlas.")
if failures:
    print(f"⚠️ {len(failures)} records failed to insert. Check error logs.")

# Optional: Save failed records to a log file
if failures:
    with open("mongo_insert_errors.log", "w") as log_file:
        for failure in failures:
            log_file.write(f"Record {failure['index']} - Error: {failure['error']}\n")
    print("⚠️ Failed records have been logged in 'mongo_insert_errors.log'.")


🚀 Starting data insertion into MongoDB...

✅ Processing 1 of 2816... Inserted successfully.
✅ Processing 2 of 2816... Inserted successfully.
✅ Processing 3 of 2816... Inserted successfully.
✅ Processing 4 of 2816... Inserted successfully.
✅ Processing 5 of 2816... Inserted successfully.
✅ Processing 6 of 2816... Inserted successfully.
✅ Processing 7 of 2816... Inserted successfully.
✅ Processing 8 of 2816... Inserted successfully.
✅ Processing 9 of 2816... Inserted successfully.
✅ Processing 10 of 2816... Inserted successfully.
✅ Processing 11 of 2816... Inserted successfully.
✅ Processing 12 of 2816... Inserted successfully.
✅ Processing 13 of 2816... Inserted successfully.
✅ Processing 14 of 2816... Inserted successfully.
✅ Processing 15 of 2816... Inserted successfully.
✅ Processing 16 of 2816... Inserted successfully.
✅ Processing 17 of 2816... Inserted successfully.
✅ Processing 18 of 2816... Inserted successfully.
✅ Processing 19 of 2816... Inserted successfully.
✅ Processing 20

In [10]:
import json
from pymongo import MongoClient
from bson import ObjectId

# Fetch the 10 most recently inserted documents
docs = list(collection.find().sort("_id", -1).limit(10))

# Recursive serializer that skips 'meta.date'
def serialize(obj):
    if isinstance(obj, ObjectId):
        return str(obj)
    if isinstance(obj, dict):
        return {
            k: serialize(v)
            for k, v in obj.items()
            if not (k == "date" and isinstance(obj, dict) and obj.get("date") == v and "meta" in obj)
        }
    if isinstance(obj, list):
        return [serialize(item) for item in obj]
    return obj

# Convert to JSON
json_output = json.dumps([serialize(doc) for doc in docs], indent=2)

# Print JSON output
print(json_output)

TypeError: Object of type datetime is not JSON serializable

In [13]:
latest_docs_cursor = collection.find(
    {"validation": {"$exists": False}}
).sort("_id", -1).limit(1642)

# Convert cursor to list to get length and reuse it
latest_docs = list(latest_docs_cursor)

print(f"The length of the latest documents: {len(latest_docs)}")

latest_ids = [doc["_id"] for doc in latest_docs]

# Step 3: Delete only those documents
if latest_ids:
    result = collection.delete_many({"_id": {"$in": latest_ids}})
    print(f"Deleted {result.deleted_count} documents.")
else:
    print("No documents matched the deletion criteria.")

The length of the latest documents: 1642
Deleted 1642 documents.


In [ ]:
{
    "sources": [
        {"source": "electrive_battery", "two_digit_ID": 11},
        {"source": "energy_voice_hydrogen", "two_digit_ID": 12},
        {"source": "offshoreWindBiz", "two_digit_ID": 13},
        {"source": "power_technology", "two_digit_ID": 14},
        {"source": "pv_magazine_hydrogen", "two_digit_ID": 15},
        {"source": "pv_magazine_manufacturing", "two_digit_ID": 16},
        {"source": "pv_magazine_utility", "two_digit_ID": 17},
        {"source": "pv_tech", "two_digit_ID": 18},
        {"source": "renewsBiz_offshore_wind", "two_digit_ID": 19},
        {"source": "renewsBiz_onshore_wind", "two_digit_ID": 21},
        {"source": "renewsBiz_solar", "two_digit_ID": 22},
        {"source": "world_energy_battery", "two_digit_ID": 23},
        {"source": "world_energy_hydrogen", "two_digit_ID": 24},
        {"source": "world_energy_solar", "two_digit_ID": 25},
        {"source": "world_energy_wind", "two_digit_ID": 26},
        {"source": "electrive_automobile", "two_digit_ID": 27},
        {"source": "pv_magazine_balance-of-system", "two_digit_ID": 28},
        {"source": "pv_magazine_energy-storage", "two_digit_ID": 29},
        {"source": "world_energy_geothermal", "two_digit_ID": 30},
        {"source": "world_energy_hydropower", "two_digit_ID": 31},
        {"source": "world_energy_nuclear", "two_digit_ID": 32},
        {"source": "world_energy_vehicles", "two_digit_ID": 33}
    ]
}